In [ ]:
from Utils import NER_Utils
import torch
from transformers import RobertaForTokenClassification, RobertaTokenizerFast, TrainingArguments, Trainer
from Reader import obtain_dataset

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    device = torch.device('cuda')
print("Current Device:", torch.cuda.current_device(), torch.cuda.get_device_name(torch.cuda.current_device()))

In [ ]:
datasets, label_list, label2id, id2label = obtain_dataset("OzRock", "BIO")

In [ ]:
# Load tokenizer and model
model_name = 'roberta-base'
tokenizer = RobertaTokenizerFast.from_pretrained(model_name, add_prefix_space=True, use_fast=True)
model = RobertaForTokenClassification.from_pretrained(model_name, num_labels=len(label_list), label2id=label2id, id2label=id2label)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results/Geo-NER",
    logging_dir="./logs/Geo-NER",
    evaluation_strategy="steps",
    save_strategy="steps",
    logging_steps=500,
    num_train_epochs=1,
    save_total_limit=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

In [ ]:
utils = NER_Utils(tokenizer, label_list)
datasets = utils.tokenize_datasets(datasets)

In [ ]:
datasets

In [ ]:
geo_ner = Trainer(
    model=model,
    args=training_args,
    compute_metrics=utils.compute_metrics,
    data_collator=utils.data_collator,
    tokenizer=tokenizer,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
)

In [ ]:
geo_ner.train()

In [ ]:
geo_ner.evaluate(datasets["eval"])

In [ ]:
#geo_ner.save_model("./results/Geo-NER/model")
geo_ner.tokenizer.save_pretrained("results/Geo-NER/checkpoint-1000")


In [ ]:
encodings=tokenizer(list(datasets["eval"]["tokens"]), padding=True, truncation=True, return_tensors="pt", is_split_into_words=True)
encodings.to(device)
with torch.no_grad():
    outputs = geo_ner.model(**encodings)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

In [ ]:
with torch.no_grad():
    outputs = geo_ner.model(**encodings)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

In [ ]:
from seqeval.metrics import classification_report
import numpy as np

In [ ]:

true_labels = []
predicted_labels = []
encodings.to("cpu")
for i in range(len(predictions)):
    pred_ids = predictions[i].cpu().numpy()
    label_ids = datasets["eval"]["label"][i]

    # Match only actual tokens (ignore padding)
    word_ids = encodings.word_ids(batch_index=i)
    aligned_preds = []
    aligned_labels = []

    previous_word_idx = None
    for j, word_idx in enumerate(word_ids):
        if word_idx is None or word_idx == previous_word_idx:
            continue  # skip subwords and special tokens
        aligned_preds.append(label_list[pred_ids[j]])
        aligned_labels.append(label_list[label_ids[word_idx]])
        previous_word_idx = word_idx

    predicted_labels.append(aligned_preds)
    true_labels.append(aligned_labels)
print(classification_report(true_labels, predicted_labels))
